# FIG04 · 局部化学状态与神经表征

这一页从化学输入、family 权重和神经坐标重建最后一张 poster 图，同时保留选择过程与失败结果。目标是说明局部化学变化怎样对应可读的神经配置，不假设全部细菌共享一个规律。

两列有不同来源：**Bacteroides** 保留先前固定 ADF−ASH 目标选出的 14 项状态；**Bifidobacterium** 使用全部 13 个坐标选出的 21 项状态。后者散点的纵轴由全数据拟合，因此是描述性投影，不能当作独立发现。下方均显示全部 13 个神经元的**实测**组均值。

重跑范围：读取既有数值输入，重建这两个完整样本拟合与已失败的 Bacteroides 全向量拟合，验证输出一致；展示一个训练折。完整留出结果读取缓存，不重跑模型网格或旧 notebook。

## 1. 环境、来源和边界

从仓库内任意目录启动。路径统一由 `poster_analysis.paths` 解析；没有外置硬盘绝对路径。方法函数操作 DataFrame，绘图函数接收已计算的表，导出只写入 `poster/`。固定数据输入的来源和 SHA-256 会写到本 notebook 对应的表中。

In [ ]:
from pathlib import Path
import sys
import hashlib
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

start = Path.cwd().resolve()
ROOT = next(p for p in (start, *start.parents)
            if (p / 'poster/src/poster_analysis').is_dir() and (p / 'reports').is_dir())
sys.path.insert(0, str(ROOT / 'poster/src'))
from poster_analysis.paths import find_repo_root, report_path, figure_dir, table_dir
from poster_analysis.constants import NEURONS, SEED
from poster_analysis.style import apply_style, save_figure
from poster_analysis import local_states as local
from poster_analysis.chemical_axes import fit_axes, transform_axes

ROOT = find_repo_root(ROOT)
apply_style()
np.random.seed(SEED)  # The calculations below are deterministic.
pd.set_option('display.max_colwidth', 75)

In [ ]:
inputs = local.load_local_inputs(
    report_path('chemistry', ROOT),
    report_path('local_poster', ROOT),
    report_path('fixed_contrast', ROOT),
)
X, metadata, unit, context = (inputs[k] for k in ['log2', 'metadata', 'unit', 'context'])
source_index = pd.DataFrame([
    {'input': name, 'path': str(path.relative_to(ROOT)),
     'sha256': hashlib.sha256(path.read_bytes()).hexdigest()}
    for name, path in sorted(inputs['source_paths'].items())
])
display(pd.DataFrame({'rows': [len(X), len(unit), len(metadata)],
                      'columns': [X.shape[1], unit.shape[1], metadata.shape[1]]},
                     index=['log2 concentration', 'unit neural coefficients', 'annotation metadata']))
display(source_index[['input', 'path']])
assert X.shape == (106, 162) and unit.shape == (106, 13)
assert X.index.is_unique and set(X.index) == set(unit.index) == set(context.index)
assert list(unit.columns) == list(NEURONS)
assert np.isfinite(X.to_numpy()).all() and np.isfinite(unit.to_numpy()).all()
np.testing.assert_allclose(np.linalg.norm(unit, axis=1), 1, atol=1e-12)

`X` 是新报告中 106 株均有有限正浓度且 QC RSD ≤ 0.30 的 162 项注释，单位变换为 `log2(c / (1 ng/mL))`；没有加 1 或插补。`unit` 是固定时间模板有符号系数的 13 维单位向量，不是原始钙曲线，也不把每个神经元重新标准化。

只按覆盖数保留至少 10 株的属，因此得到 29 株 Bacteroides 和 11 株 Bifidobacterium。这个门槛使描述性三等分每组至少 3 株，不是统计功效保证。所有入选属成员和原分类备注均保留。

In [ ]:
coverage = context.groupby('genus').size().rename('n_strains').sort_values(ascending=False)
eligible = coverage[coverage >= 10]
display(eligible.to_frame())
assert set(eligible.index) == set(local.GENERA)
ids = {genus: sorted(context.index[context.genus.eq(genus)]) for genus in local.GENERA}
x_bact, x_bif = X.loc[ids['Bacteroides']], X.loc[ids['Bifidobacterium']]
y_bact, y_bif = unit.loc[x_bact.index], unit.loc[x_bif.index]
display(context.loc[x_bact.index].loc[lambda frame: frame.taxonomy_note.fillna('').ne(''),
                                      ['species', 'taxonomy_note']])

## 2. 化学状态怎样算出来

先用属内训练样本的均值和样本 SD（`ddof=1`）把每项 log2 浓度变成 z。这里为了重现完整样本描述，训练集就是这个属的全部样本；后面的单折演示会只用训练行。

随后按 Pearson 距离 `1−r`、average linkage、固定切割距离 0.5 分组。合格组至少有 3 项注释及 3 个 Mass-column family。`chemical_axes.py` 保留原来验证过的纯化学函数，不读取神经响应或历史组定义。

In [ ]:
# An explicit log2 → z step, illustrated with Bifidobacterium.
z_bif_all, bif_means, bif_sds = local.standardize_chemistry(x_bif)
display(pd.DataFrame({'log2_mean': bif_means, 'sample_sd': bif_sds}).head(6))
display(z_bif_all.iloc[:3, :6])
np.testing.assert_allclose(z_bif_all.mean(), 0, atol=1e-12)
np.testing.assert_allclose(z_bif_all.std(ddof=1), 1, atol=1e-12)

axes_bif = fit_axes(x_bif, metadata)
chemical_groups = pd.DataFrame([
    {'state_id': state, 'n_annotations': len(members),
     'n_families': metadata.loc[members, 'family'].nunique()}
    for state, members in axes_bif['module_members'].items()
])
display(chemical_groups)

每个 family 先获等权，再把该 family 权重均分给其中的注释。这样同一个 Mass-column family 的重复注释不会获得额外总权重，但 family 也不等同于已经确认的独立分子。

一个状态的分数就是 `score = Σ(weight × z)`；不会再把这个总分除以其 SD。下面先以纯化学分组中的第一组演示，而不使用任何神经值选该例。

In [ ]:
example_state = next(iter(axes_bif['module_members']))
example_members = axes_bif['module_members'][example_state]
example_weights = local.family_weights(metadata.loc[example_members])
weight_audit = metadata.loc[example_members, ['family']].assign(weight=example_weights)
display(weight_audit.head(8))
family_totals = weight_audit.groupby('family').weight.sum()
np.testing.assert_allclose(family_totals, 1 / len(family_totals), atol=1e-12)
manual_example_score = (z_bif_all[example_members] * example_weights).sum(axis=1)
np.testing.assert_allclose(manual_example_score, axes_bif['train_scores'][example_state], atol=1e-12)
display(axes_bif['train_scores'].round(3))

## 3. 从候选化学分数到神经选择

对于一个候选分数 `s`，每个神经坐标只拟合截距和一条斜率：`u_j = intercept_j + slope_j × s`。13 个坐标使用已有尺度，因此合计 SSE 表达的是现有神经表示的拟合误差，并不保证每个神经元有相等的生物学重要性。

Bifidobacterium 用全部 13 坐标的训练 SSE 选一个状态。这里重现已执行的固定规则，不扩大候选或模型族。选择后的相关、IQR 效应、散点和组均值均属于条件于全数据选择的描述。

In [ ]:
# A minimal visible OLS example before calling the complete candidate selector.
candidate_score = axes_bif['train_scores'][example_state]
candidate_ols = local.fit_ols(candidate_score, y_bif)
design = np.column_stack([np.ones(len(candidate_score)), candidate_score])
beta_direct = np.linalg.lstsq(design, y_bif.to_numpy(), rcond=None)[0]
np.testing.assert_allclose(beta_direct[0], candidate_ols.intercept, atol=1e-12)
np.testing.assert_allclose(beta_direct[1], candidate_ols.slope, atol=1e-12)
display(candidate_ols[['intercept', 'slope', 'sse_model', 'sse_mean']])

fit_bif = local.select_state(x_bif, metadata, y_bif, objective='all13')
display(fit_bif['candidates'].drop(columns='members_json').sort_values('sse_model_total'))
assert fit_bif['state_id'] == 'L02'

Bacteroides 的主图保留**先前固定的 ADF−ASH 目标**，从原 162 项按同一化学规则重新分组后，以训练相关的平方选组。它选中 14 项 L03。

同时保留这次完整 13 维目标的失败尝试：它改选 3 项 L12，内部留出 SSE 比训练均值基线更差。不能因为它表观拟合稍好就替换原锚点，也不能把两种目标的改善百分比直接排名。

In [ ]:
fit_bact = local.select_state(x_bact, metadata, y_bact, objective='adf_minus_ash')
failed_bact_population = local.select_state(x_bact, metadata, y_bact, objective='all13')
selection_summary = pd.DataFrame([
    {'case': 'Bacteroides: previous contrast', 'objective': 'ADF-ASH',
     'state': fit_bact['state_id'], 'annotations': len(fit_bact['axes']['module_members'][fit_bact['state_id']])},
    {'case': 'Bifidobacterium: population', 'objective': 'all 13 coordinates',
     'state': fit_bif['state_id'], 'annotations': len(fit_bif['axes']['module_members'][fit_bif['state_id']])},
    {'case': 'Bacteroides: failed population extension', 'objective': 'all 13 coordinates',
     'state': failed_bact_population['state_id'],
     'annotations': len(failed_bact_population['axes']['module_members'][failed_bact_population['state_id']])},
])
display(selection_summary)
assert fit_bact['state_id'] == 'L03' and failed_bact_population['state_id'] == 'L12'
bact_members = fit_bact['axes']['module_members'][fit_bact['state_id']]
bif_members = fit_bif['axes']['module_members'][fit_bif['state_id']]
assert set(bact_members) == set(inputs['bact_members'].metabolite)
assert set(bif_members) == set(inputs['bif_members'].query('genus == "Bifidobacterium"').metabolite)
display(failed_bact_population['candidates'].drop(columns='members_json').sort_values('sse_model_total').head(4))

## 4. 查看状态的化学组成和一个样本的分数贡献

两属的状态不是同一套化学物。完整成员及原报告分类保留在表中，示例成分不能据此解释为致效物或通路。下图只检查获选 Bifidobacterium 化学成员自身的共变结构。

In [ ]:
selected_members = pd.concat([
    metadata.loc[bact_members].assign(genus='Bacteroides', weight=local.family_weights(metadata.loc[bact_members])),
    metadata.loc[bif_members].assign(genus='Bifidobacterium', weight=local.family_weights(metadata.loc[bif_members])),
]).reset_index()
display(selected_members[['genus', 'metabolite', 'family', 'weight', 'Class']])

z_bif_selected = z_bif_all[bif_members]
fig_covariance = local.plot_member_covariance(z_bif_selected, 'Bifidobacterium: selected chemical annotations')
save_figure(fig_covariance, figure_dir('supporting', ROOT) / 'fig04_bifidobacterium_member_covariance')
plt.show()

下面选择化学分数排序的中间一株，只用于演示加权求和。条形表示该株每个成员的 `z × weight`，正负取决于相对该属均值的浓度。贡献相加严格等于分数；它不是各物质对神经响应的独立贡献。

In [ ]:
bif_rank = local.rank_thirds(fit_bif['score'])
example_strain = bif_rank.sort_values('chemical_rank').index[len(bif_rank) // 2]
bif_weights = local.family_weights(metadata.loc[bif_members])
point_contributions = pd.DataFrame({
    'z': z_bif_selected.loc[example_strain], 'weight': bif_weights,
    'weighted_contribution': z_bif_selected.loc[example_strain] * bif_weights,
})
np.testing.assert_allclose(point_contributions.weighted_contribution.sum(), fit_bif['score'].loc[example_strain], atol=1e-12)
display(point_contributions.assign(strain=example_strain))
fig_contributions = local.plot_score_contributions(z_bif_selected.loc[example_strain], bif_weights, example_strain)
save_figure(fig_contributions, figure_dir('supporting', ROOT) / 'fig04_example_score_contributions')
plt.show()

## 5. 从每株数据到实测的 13 神经元组均值

分组只按已选化学分数及菌株 ID 排序，用 `np.array_split` 得到 Bacteroides 的 10/10/9 和 Bifidobacterium 的 4/4/3 株。这是人数相近的排名组，不是等宽浓度区间，也没有检验阈值机制。

先直接计算各组的**实测**神经均值，再减去该属全体均值。所有 13 行、两个属使用相同神经色阶，没有行 z-score。由于化学轴先由同一批神经响应选出，这些组差异仍是描述性结果。

In [ ]:
bact_rank = local.rank_thirds(fit_bact['score'])
observed_bif_groups = local.observed_group_means(y_bif, bif_rank)
display(pd.concat([bact_rank.rank_group.value_counts().rename('Bacteroides'),
                   bif_rank.rank_group.value_counts().rename('Bifidobacterium')], axis=1).loc[local.GROUPS])
display(observed_bif_groups.pivot(index='neuron', columns='rank_group', values='centered_mean').loc[list(NEURONS), local.GROUPS])

bact_display = local.state_display(x_bact, metadata, y_bact, context, fit_bact, 'Bacteroides')
bif_display = local.state_display(x_bif, metadata, y_bif, context, fit_bif, 'Bifidobacterium')
display_tables = local.combine_display_tables([bact_display, bif_display])
parity = local.compare_display_tables(display_tables, inputs['reference_display'])
display(parity)

Bifidobacterium 散点使用完整 13 维斜率向量的单位方向：`(unit − genus_mean) @ (slope / ||slope||)`。这个方向与化学关系一起从全数据拟合，不能把散点相关强度解释为独立的 13 神经元证据。其意义需结合下面未投影的 13 个实测均值及留出误差看。

IQR 图显示同一化学分数跨一个观测 IQR 时各神经坐标的拟合变化。Bifidobacterium 主要涉及 AWCOFF、AWA、ADF、ASH；完整向量受益不等于全部 13 个坐标均有正的预测信息。

In [ ]:
bif_direction = fit_bif['model'].slope / np.linalg.norm(fit_bif['model'].slope)
projection = (y_bif - y_bif.mean()) @ bif_direction
np.testing.assert_allclose(projection, bif_display['strain_scores'].set_index('strain').plot_response.loc[y_bif.index], atol=1e-12)
display(display_tables['neural_slopes'][['genus', 'neuron', 'iqr_effect']].pivot(index='neuron', columns='genus', values='iqr_effect').loc[list(NEURONS)])
fig_effects = local.plot_neural_effects(display_tables['neural_slopes'])
save_figure(fig_effects, figure_dir('supporting', ROOT) / 'fig04_neural_iqr_effects')
plt.show()

## 6. 内部留出检查：读取缓存，并演示一个训练折

留一菌株与留一记录菌种的旧运行在每折内重新计算化学尺度、分组、成员、权重、择优和神经系数。汇总指标为 `1 − SSE_model / SSE_training_mean`，不是 RMSE 降幅。模板表示和全局化学 QC 已固定，故这是条件于该表示的内部检查，不是独立实验验证。

下表区分三个任务。Bacteroides 原 ADF−ASH 的改善约 23.1% / 20.0%；Bifidobacterium 全 13 维约 15.9% / 18.2%；Bacteroides 新全 13 维为 −4.8% / −15.7%。这些比例的预测目标不同。

In [ ]:
population_performance = inputs['population_performance']
contrast_performance = inputs['contrast_performance']
population_summary = population_performance.query('neuron == "all13"')[['genus', 'scheme', 'error_improvement']].copy()
population_summary['task'] = population_summary.genus + ': all13 objective'
contrast_summary = contrast_performance.query('target == "primary_unit_adf_minus_ash"')[['scheme', 'error_improvement']].copy()
contrast_summary['task'] = 'Bacteroides: fixed ADF-ASH objective'
performance_summary = pd.concat([population_summary[['task', 'scheme', 'error_improvement']], contrast_summary], ignore_index=True)
display(performance_summary.pivot(index='task', columns='scheme', values='error_improvement').mul(100).round(2))

# Independent arithmetic check of all cached population SSE summaries; no refit.
cached_predictions = inputs['population_predictions']
performance_checks = []
for (genus, scheme), block in cached_predictions.groupby(['genus', 'scheme']):
    for neuron, subset in [('all13', block)] + list(block.groupby('neuron')):
        sse = ((subset.observed - subset.model_prediction) ** 2).sum()
        baseline = ((subset.observed - subset.training_mean_prediction) ** 2).sum()
        cached = population_performance.query('genus == @genus and scheme == @scheme and neuron == @neuron').iloc[0]
        np.testing.assert_allclose([sse, baseline, 1 - sse / baseline],
                                   [cached.sse_model, cached.sse_training_mean, cached.error_improvement], atol=1e-12)
        performance_checks.append({'genus': genus, 'scheme': scheme, 'neuron': neuron, 'error_improvement': 1 - sse / baseline})
print(f'Checked {len(performance_checks)} cached metric rows from saved predictions.')

# Check the historical contrast metric from its own saved predictions too.
contrast_checks = []
target = 'primary_unit_adf_minus_ash'
for scheme, block in inputs['contrast_predictions'].groupby('scheme'):
    sse = ((block[target] - block['pred_' + target]) ** 2).sum()
    baseline = ((block[target] - block['baseline_' + target]) ** 2).sum()
    cached = contrast_performance.query('scheme == @scheme and target == @target').iloc[0]
    np.testing.assert_allclose([sse, baseline, 1 - sse / baseline],
                               [cached.sse_model, cached.sse_training_mean_baseline, cached.error_improvement], atol=1e-12)
    contrast_checks.append({'scheme': scheme, 'error_improvement': 1 - sse / baseline})
display(pd.DataFrame(contrast_checks))

In [ ]:
fig_cv = local.plot_cached_performance(population_performance, 'Bifidobacterium')
save_figure(fig_cv, figure_dir('supporting', ROOT) / 'fig04_bifidobacterium_leaveout_by_neuron')
plt.show()
fold_stability = inputs['population_fold_selection'].groupby(['genus', 'scheme']).agg(
    n_folds=('fold_id', 'size'), exact_member_matches=('full_members_exact_match', 'sum'),
    median_member_jaccard=('full_member_jaccard', 'median'), minimum_member_jaccard=('full_member_jaccard', 'min'))
display(fold_stability)

正向留出信号集中在部分坐标，尤其 AWCOFF 和 AWA；不少其他坐标比训练均值基线差。Bifidobacterium 的完整 21 项成员表也不会在每折原样出现：中位 Jaccard 约 0.77，因此不把这一固定名单说成已独立验证。

下面仅取 Bifidobacterium **按名称排序的第一个记录菌种**做一折演示。训练数据重新决定均值、SD、分组和候选；留出行只应用训练参数。演示选择基于确定性顺序，未按预测好坏选择。预期与相应缓存折完全一致。

In [ ]:
species_bif = context.loc[x_bif.index, 'species']
heldout_species = sorted(species_bif.unique())[0]
test_ids = species_bif.index[species_bif.eq(heldout_species)].tolist()
train_ids = species_bif.index[~species_bif.eq(heldout_species)].tolist()
demo = local.fit_one_fold(x_bif, metadata, y_bif, train_ids, test_ids, objective='all13')
assert set(demo['fitted']['axes']['train_ids']) == set(train_ids)
assert not set(train_ids).intersection(test_ids)
np.testing.assert_allclose(demo['fitted']['axes']['means'], x_bif.loc[train_ids].mean(), atol=1e-12)
np.testing.assert_allclose(demo['fitted']['axes']['scales'], x_bif.loc[train_ids].std(ddof=1), atol=1e-12)
cached_demo = cached_predictions.query('genus == "Bifidobacterium" and scheme == "leave_one_recorded_species_out"').query('strain in @test_ids')
cached_wide = cached_demo.pivot(index='strain', columns='neuron', values='model_prediction').loc[test_ids, list(NEURONS)]
np.testing.assert_allclose(demo['predictions'], cached_wide, atol=1e-12)
demo_comparison = pd.concat({'observed': demo['observed'], 'predicted': demo['predictions'], 'training_mean': demo['baseline']}, names=['value_type'])
display(pd.DataFrame({'heldout_species': [heldout_species], 'n_train': [len(train_ids)],
                      'n_test': [len(test_ids)], 'selected_state': [demo['fitted']['state_id']]}))
display(demo_comparison)

## 7. 最终主图与全部个体支持

左列保留 Bacteroides 的可读局部对比；右列扩展到 Bifidobacterium 的完整神经配置。所有 40 株均保留，记录日期只在元信息里。主图不承诺普适规律、因果、剂量关系或行为意义。化学命名来自报告注释，图中的例子不是机制归因。

主图四个面板分别是：两个属的逐株对应，以及各自三个化学排名组的 13 维实测偏移。随后两个支持图逐株展示全部成员与全部神经坐标。

In [ ]:
fig_main = local.plot_main(display_tables)
main_exports = save_figure(fig_main, figure_dir('main', ROOT) / 'fig04_local_chemical_states')
plt.show()

In [ ]:
for genus in local.GENERA:
    fig = local.plot_individual_support(display_tables, genus)
    save_figure(fig, figure_dir('supporting', ROOT) / f'fig04_individuals_{genus.lower()}')
    plt.show()

## 8. 导出、核对与本页结论

本 notebook 保存计算中间表和最终绘图表，便于逐步修改与核对。可复用方法位于 `poster/src/poster_analysis/local_states.py` 与 `chemical_axes.py`；完整历史折参数仍在来源报告，不在本页重跑。

当前图给神经表征一个局部的化学对应：Bacteroides 的 ADF−ASH 对比以及 Bifidobacterium 中若干共同变化坐标。它保留 13 维观测，同时清楚显示信息并非平均分布在全部坐标。Bacteroides 全向量扩展的负结果、全数据择优和固定表示下内部检查的限制都保留在上文。

In [ ]:
destination = table_dir(ROOT)
for name, frame in display_tables.items():
    frame.to_csv(destination / f'fig04_{name}.csv', index=False)
source_index.to_csv(destination / 'fig04_source_index.csv', index=False)
selection_summary.to_csv(destination / 'fig04_selection_summary.csv', index=False)
performance_summary.to_csv(destination / 'fig04_performance_summary.csv', index=False)
pd.DataFrame(performance_checks).to_csv(destination / 'fig04_recomputed_cached_performance.csv', index=False)
pd.DataFrame(contrast_checks).to_csv(destination / 'fig04_recomputed_contrast_performance.csv', index=False)
parity.to_csv(destination / 'fig04_display_parity.csv', index=False)
point_contributions.assign(strain=example_strain).to_csv(destination / 'fig04_example_score_contributions.csv')
demo_comparison.to_csv(destination / 'fig04_one_fold_demo.csv')
for name, fit in [('bacteroides_contrast', fit_bact), ('bifidobacterium_all13', fit_bif),
                  ('bacteroides_failed_all13', failed_bact_population)]:
    fit['candidates'].to_csv(destination / f'fig04_candidates_{name}.csv', index=False)
for row in source_index.itertuples():
    assert hashlib.sha256((ROOT / row.path).read_bytes()).hexdigest() == row.sha256
checks = pd.DataFrame([
    {'check': 'all source hashes unchanged', 'passed': True},
    {'check': 'all five figure-data tables match the accepted figure', 'passed': True},
    {'check': '56 cached population metric rows independently recomputed', 'passed': len(performance_checks) == 56},
    {'check': '2 historical contrast metric rows independently recomputed', 'passed': len(contrast_checks) == 2},
    {'check': 'one train-only species fold matches cache', 'passed': True},
    {'check': 'all 40 strains and 13 neurons retained', 'passed': len(display_tables['strain_scores']) == 40},
])
assert checks.passed.all()
checks.to_csv(destination / 'fig04_checks.csv', index=False)
display(checks)
display(pd.DataFrame({'main_export': [str(path.relative_to(ROOT)) for path in main_exports]}))